In [2]:
import sys, os
sys.path.append("../src")
import paths
from paths import CKPT_DIR, DATA_ROOT

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from model import GPT, GPTConfig
from dataset import get_batch

device = "cuda"

# 내 모델
ckpt = torch.load(os.path.join(CKPT_DIR, "sft30m", "best.pt"), map_location="cpu", weights_only=False)
mine = GPT(GPTConfig(**ckpt["model_config"]))
mine.load_state_dict(ckpt["model"])
mine.to(device).eval()

# HF 모델 (방금 저장한 폴더에서, 졸업 프로젝트와 같은 방식으로 로드)
hf_dir = os.path.join(DATA_ROOT, "hf", "sft30m")
hf = AutoModelForCausalLM.from_pretrained(hf_dir).to(device).eval()
print(type(hf).__name__, "/ tied:", hf.lm_head.weight is hf.transformer.wte.weight)

# 같은 입력으로 비교
x, y = get_batch("val", 4, 256, device)                  # (4, 256)
with torch.no_grad():
    logits_mine, loss_mine = mine(x, y)                  # (4, 256, 50257)
    out = hf(x, labels=x)                                # HF는 labels를 내부에서 한 칸 밀어서 loss 계산
    logits_hf = out.logits                               # (4, 256, 50257)

diff = (logits_mine - logits_hf).abs()
print(f"logits 최대 차이: {diff.max().item():.2e} / 평균 차이: {diff.mean().item():.2e}")
print("다음 토큰 예측(argmax) 일치율:", (logits_mine.argmax(-1) == logits_hf.argmax(-1)).float().mean().item())
print(f"loss  내 모델 {loss_mine.item():.4f} / HF {out.loss.item():.4f}")

c:\dev\mini-llm\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 76/76 [00:00<00:00, 7725.63it/s]


GPT2LMHeadModel / tied: True


[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


logits 최대 차이: 2.48e-05 / 평균 차이: 2.23e-06
다음 토큰 예측(argmax) 일치율: 1.0
loss  내 모델 1.7374 / HF 1.7359


In [2]:
with torch.no_grad():
    out32 = hf(x, output_hidden_states=True)            # 블록마다 출력(잔차 통로의 x)을 함께 반환

# hidden_states: (임베딩 출력, 블록1 출력, ..., 블록6 출력) → 7개, 각 (B, T, C)
for i, h in enumerate(out32.hidden_states):
    name = "임베딩" if i == 0 else f"블록 {i}"
    print(f"{name:6s}  최대 |값| {h.abs().max().item():9.2f}")
print(f"logits  최대 |값| {out32.logits.abs().max().item():9.2f}")
print("fp16 한계: 65504")

임베딩     최대 |값|      0.45
블록 1    최대 |값|      8.78
블록 2    최대 |값|     10.56
블록 3    최대 |값|     11.37
블록 4    최대 |값|     11.90
블록 5    최대 |값|     13.45
블록 6    최대 |값|     21.53
logits  최대 |값|     25.68
fp16 한계: 65504


In [3]:
import copy

hf16 = copy.deepcopy(hf).half()                          # 파라미터를 전부 fp16으로 변환한 복사본
with torch.no_grad():
    logits32 = hf(x).logits.float()                      # (4, 256, V)
    logits16 = hf16(x).logits.float()                    # (4, 256, V) 비교를 위해 fp32로 올림

print("NaN 있음:", torch.isnan(logits16).any().item(), "/ inf 있음:", torch.isinf(logits16).any().item())
d = (logits32 - logits16).abs()
print(f"최대 차이 {d.max().item():.3e} / 평균 차이 {d.mean().item():.3e}")
print("argmax 일치율:", (logits32.argmax(-1) == logits16.argmax(-1)).float().mean().item())

ce = torch.nn.functional.cross_entropy
l32 = ce(logits32[:, :-1].reshape(-1, logits32.size(-1)), x[:, 1:].reshape(-1)).item()
l16 = ce(logits16[:, :-1].reshape(-1, logits16.size(-1)), x[:, 1:].reshape(-1)).item()
print(f"loss  fp32 {l32:.4f} / fp16 {l16:.4f}")

NaN 있음: False / inf 있음: False
최대 차이 5.397e-02 / 평균 차이 3.983e-03
argmax 일치율: 0.9990234375
loss  fp32 2.2270 / fp16 2.2271


In [4]:
tok = AutoTokenizer.from_pretrained(hf_dir)
prompts = ["Once upon a time", "Tom and Sue went to the park.", "The little dog was sad because"]

for p in prompts:
    enc = tok(p, return_tensors="pt").to(device)
    with torch.no_grad():
        g32 = hf.generate(**enc, max_new_tokens=100, do_sample=False, pad_token_id=50256)    # (1, T+100)
        g16 = hf16.generate(**enc, max_new_tokens=100, do_sample=False, pad_token_id=50256)
    a, b = g32[0].tolist(), g16[0].tolist()
    n = min(len(a), len(b))
    first_diff = next((i for i in range(n) if a[i] != b[i]), None)
    print(f"\n[{p}] 같음: {a == b}" + ("" if first_diff is None else f" / {first_diff}번째 토큰부터 달라짐"))
    print("fp16:", tok.decode(b, skip_special_tokens=True)[:300])


[Once upon a time] 같음: True
fp16: Once upon a time, there was a little girl named Lily. She loved to play with her toys and her favorite toy was a teddy bear named Teddy. One day, Lily's mom asked her to help with the laundry. Lily was happy to help and started to put the clothes in the washing machine.

Suddenly, Lily's little brot

[Tom and Sue went to the park.] 같음: True
fp16: Tom and Sue went to the park. They saw a big slide. They wanted to go on the slide. They ran to the slide. They took turns to go down. They laughed and shouted.

But then they saw a boy. He was bigger than them. He had a red shirt and a blue shirt. He looked mean. He was making noise.

"Hey, that's 

[The little dog was sad because] 같음: True
fp16: The little dog was sad because he wanted to play with the ball. He tried to jump over the fence, but he was too small. He could not reach the ball.

Then, a kind girl saw the little dog and wanted to help. She picked up the ball and gave it to the dog. The dog was h

In [3]:
import numpy as np

np.random.seed(0)                                               # 같은 배치를 다시 뽑도록 고정
batches = [get_batch("val", 16, 256, device) for _ in range(10)]   # 10 × (16, 256)

for run in ["gpt30m", "sft30m"]:
    m = AutoModelForCausalLM.from_pretrained(os.path.join(DATA_ROOT, "hf", run)).to(device).eval()
    with torch.no_grad():
        losses = [m(x, labels=x).loss.item() for x, _ in batches]
    print(f"{run}: 사전학습 val 데이터 loss {sum(losses) / len(losses):.4f}")
    del m

Loading weights: 100%|██████████| 76/76 [00:00<00:00, 520.44it/s]


gpt30m: 사전학습 val 데이터 loss 1.5859


Loading weights: 100%|██████████| 76/76 [00:00<00:00, 1100.95it/s]


sft30m: 사전학습 val 데이터 loss 1.9487
